# Train CIFAR-100 on Kaggle

Trains the project's CNN or ImageNet-pretrained MobileNetV2 using `CIFAR100/CIFAR100/{TRAIN,VALID,TEST}`. The best model and class names are saved to `/kaggle/working`. Set the Kaggle notebook accelerator to GPU to use a GPU.


In [ ]:
from pathlib import Path
import json
import tensorflow as tf
from tensorflow.keras import layers

# Use GPU memory as needed instead of reserving it all at startup.
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print('GPUs:', gpus if gpus else 'No GPU detected; using CPU')

tf.keras.utils.set_random_seed(42)
IMAGE_SIZE = (32, 32)
NUM_CLASSES = 100
BATCH_SIZE = 32
EPOCHS = 20
MODEL_TYPE = 'mobilenetv2'  # Choose 'mobilenetv2' or 'cnn'
OUTPUT_DIR = Path('/kaggle/working')


## Find and load the dataset

This locates the folder containing `TRAIN`, `VALID`, and `TEST` under Kaggle's `/kaggle/input`. Each split should contain one folder per class.


In [ ]:
input_dir = Path('/kaggle/input')
split_dirs = [p.parent for p in input_dir.rglob('TRAIN')
              if p.is_dir() and (p.parent / 'VALID').is_dir()
              and (p.parent / 'TEST').is_dir()]
if not split_dirs:
    raise FileNotFoundError('Could not find CIFAR100/CIFAR100/{TRAIN,VALID,TEST} under /kaggle/input')
DATA_DIR = split_dirs[0]
print('Using dataset:', DATA_DIR)

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'TRAIN', image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    label_mode='int', shuffle=True, seed=42)
class_names = train_ds.class_names
if len(class_names) != NUM_CLASSES:
    raise ValueError(f'Expected 100 class folders, found {len(class_names)}')

valid_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'VALID', image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    label_mode='int', class_names=class_names, shuffle=False)
test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'TEST', image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    label_mode='int', class_names=class_names, shuffle=False)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
valid_ds = valid_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)


## Build the model

MobileNetV2 uses frozen ImageNet features and a new CIFAR-100 classification head. The CNN matches the compact model in `src/model.py`.


In [ ]:
def build_cnn(num_classes=NUM_CLASSES):
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
    x = layers.Rescaling(1.0 / 255, name='rescale')(inputs)
    x = layers.RandomFlip('horizontal', name='random_flip')(x)
    x = layers.RandomRotation(0.06, name='random_rotation')(x)
    for filters in (32, 64, 128):
        x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='class_scores')(x)
    model = tf.keras.Model(inputs, outputs, name='cifar100_cnn')
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def build_mobilenetv2(num_classes=NUM_CLASSES):
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='image')
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    backbone = tf.keras.applications.MobileNetV2(
        include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3))
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='class_scores')(x)
    model = tf.keras.Model(inputs, outputs, name='cifar100_mobilenetv2_transfer')
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

if MODEL_TYPE == 'mobilenetv2':
    model = build_mobilenetv2()
elif MODEL_TYPE == 'cnn':
    model = build_cnn()
else:
    raise ValueError("MODEL_TYPE must be 'mobilenetv2' or 'cnn'")
model.summary()


## Train, evaluate, and save

The best validation accuracy checkpoint is saved as a Keras model. The `TEST` split is evaluated after training.


In [ ]:
model_path = OUTPUT_DIR / f'cifar100_{MODEL_TYPE}.keras'
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        model_path, monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True),
]
history = model.fit(train_ds, validation_data=valid_ds,
                    epochs=EPOCHS, callbacks=callbacks)

best_model = tf.keras.models.load_model(model_path)
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=1)
(OUTPUT_DIR / 'class_names.json').write_text(
    json.dumps(class_names, indent=2), encoding='utf-8')
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')
print(f'Saved model to: {model_path}')
